# Séance 6 — Quantification et compromis débit/distorsion

PhL

**Thème :** Quantification scalaire, tables JPEG, erreur, qualité visuelle et compression avec pertes.


## Préambule technique

Les images sont représentées par des tableaux NumPy : niveaux de gris `(hauteur, largeur)` ou couleurs `(hauteur, largeur, 3)`.

## Objectifs et déroulé

### Objectifs
- Comprendre la quantification comme source principale de perte dans JPEG.
- Quantifier et déquantifier les coefficients DCT.
- Étudier l’erreur en fonction d’un paramètre de qualité.

### Déroulé indicatif (3 h)
1. 30 min — quantification uniforme.
2. 40 min — table de quantification et fréquence.
3. 40 min — exercices.
4. 45 min — étape projet : pipeline DCT + quantification + reconstruction.
5. 25 min — PSNR, artefacts et limites des métriques.

**Travail personnel (~2 h)** : produire une courbe qualité/erreur pour plusieurs facteurs de quantification.

## Rappels

### Quantification

La _quantification_ des coefficients DCT divise chaque coefficient $F(u,v)$ par un pas strictement positif $Q(u,v)$, puis arrondit à l'entier le plus proche :
$$
\widehat F(u,v)=\operatorname{arrondi}\left(F(u,v)/Q(u,v)\right).
$$
Pour reconstruire les coefficients "avant quantification", on multiplie par le pas : $F_r=\widehat FQ$. 

L’arrondi de la quantification introduit une approximation : plusieurs coefficients initiaux peuvent donner le même entier quantifié. 
On parle de transformation **avec perte**. 
Ainsi, la quantification inverse, dite aussi _déquantification_, ne permet pas de retrouver systématiquement les coefficients originaux.

Un pas plus grand entraîne généralement une erreur plus importante. 

### Mesurer la perte d'information

On a déjà mentionné la MSE : "Mean Square Error" (séance 4). 
Pour deux images de taille $H\times W$, l’_erreur quadratique moyenne (MSE)_ vaut
$$
\operatorname{MSE}(I,\widehat I)=\frac1{HW}\sum_{y=0}^{H-1}\sum_{x=0}^{W-1}(I(y,x)-\widehat I(y,x))^2.
$$
Elle vaut 0 pour deux images identiques et pénalise fortement les grands écarts.

En traitement du signal, le _PSNR_ (rapport signal sur bruit de crête) est un indicateur de la qualité de la transmission d'une information. 
Il se mesure, en dB (décibels), comme :
$$
\operatorname{PSNR}=10\log_{10}\left(\frac{L^2}{\operatorname{MSE}}\right),
$$
où $L$ est l’intensité maximale -- ici 255 pour des images 8 bits. 

Le PSNR est pratique pour comparer des réglages mais ne modélise pas parfaitement la perception humaine, en particulier des images.

Si la MSE vaut 0, le PSNR est infini. 
À conventions identiques, un PSNR élevé correspond à une MSE faible. 
Cependant, cela ne garantit pas une meilleure qualité visuelle : des artefacts différents (blocs, halos, perte de détails) peuvent donner le même PSNR. 

### De JPEG ... 

Le pas de la quantification peut être constant mais JPEG utilise des pas qui varient selon les fréquences $(u,v)$ :
la matrice $Q$ suivante definie chaque $Q_{u, v}$ appliqué à chaque bloc 8 x 8 de l'image après DCT :

$$ Q = (Q_{u, v}) = \left( \begin{array}{cccccccc}
                 16 & 11 & 10 & 16 & 24 & 40 & 51 & 61 \\
                 12 & 12 & 14 & 19 & 26 & 58 & 60 & 55 \\
                 14 & 13 & 16 & 24 & 40 & 57 & 69 & 56 \\
                 14 & 17 & 22 & 29 & 51 & 87 & 80 & 62 \\
                 18 & 22 & 37 & 56 & 68 & 109 & 103 & 77 \\
                 24 & 35 & 55 & 64 & 81 & 104 & 113 & 92 \\
                 49 & 64 & 78 & 87 & 103 & 121 & 120 & 101 \\
                 72 & 92 & 95 & 98 & 112 & 100 & 103 & 99
\end{array} \right).$$

La quantification après DCT conduit à de nombreux coefficients nuls.
Cette propriété est exploitée pour _réduire le stockage_ de chaque bloc et ainsi _compresser_ une image.   

JPEG exploite cette propriété procède en deux étapes pour _réduire le stockage_ de chaque bloc 8 x 8 en un vecteur de taille bien inférieure à 64 :

- réorganisation des coefficients par [codage zig-zag](https://fr.wikipedia.org/wiki/JPEG#Codage,_compression_RLE_et_Huffman), puis 
- [codage RLE](https://fr.wikipedia.org/wiki/Codage_par_plages) des suites de zéros obtenues. 

**Remarque.** Une chaîne JPEG courante comprend d'autres étapes que les 4 ici décrites (DCT, quantification, codages zig-zag et RLE) 
: conversion RGB vers luminance/chrominance, sous-échantillonnage de chrominance, découpage en blocs 8×8, DCT, quantification, réorganisation des coefficients et codage entropique. 

Retenons cependant que la quantification introduit l’essentiel de la perte dans cette compression.

### ... à un mini-JPEG

**IMPORTANT** 
Dans ce projet, nous allons nour limiter à _un "mini JPEG"_ constitué des 4 étapes.

1. Découpage en blocs 8×8, puis pour chaque bloc
2. DCT,
3. quantification avec $Q$,
4. codages zig-zag et _pseudo-RLE_.

Codage pseudo-RLE : On se limite à un codage pseudo-RLE : 
**en partant de la fin, la suite de zéros consécutifs ne sera pas stockée.**

Le _taux de compression_ de cette chaine "mini-JPEG" se mesure en comparant les tailles des fichier (images) oroginaux vs tranformés. 

La reconstruction d'une image ainsi comprimée s'effectue par la chaine de transformations inverses.

Une convention classique de JPEG centre les intensités de l'image (avec un décalage de -128) avant l'étape de DCT et effectue le décallage inverse après la DCT inverse de la reconstruction. 
Vous pouvez introduire (ou non) ce centrage à votre mini-JPEG. 


**Dans le notebook :** comparer des images de même taille et représentation ; choisir correctement $L$ ; traiter le cas MSE = 0 ; mesurer aussi la taille du fichier.

**Liens :** 

- [Quantification](https://fr.wikipedia.org/wiki/Quantification_(signal))
- [Rapport signal sur bruit](https://fr.wikipedia.org/wiki/Rapport_signal_sur_bruit)
- [Erreur quadratique moyenne](https://fr.wikipedia.org/wiki/Erreur_quadratique_moyenne)
- [JPEG](https://fr.wikipedia.org/wiki/JPEG)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

## Exercice 1 : MSE et PSNR

1. Coder les fonctions de mesure d'erreur MSE et PSNR pour des images sur [0,255].
2. Quel est le PSNR si les images sont identiques ?

In [16]:
# VOTRE CODE ICI

## Correction 1

Le MSE est nul et le PSNR est infini pour deux images identiques.

## Exercice 2 : quantification uniforme

1. Ecrire les fonctions `quantifier()` et `dequantifier()`.
2. Quantifier les coefficients `[-13, -5, 0, 6, 17]` avec un pas `Q=4`, puis déquantifier.
3. Mesurer la perte d'information avec `mse` et `psnr`. Qu'en pensez-vous ? 
4. Que pensez-vous de l’information perdue ?

In [16]:
# VOTRE CODE ICI

## Exercice 3 : quantification JPEG

1. Observer les coefficients $Q_{u,v}$ de la matrice de quantification $Q$ de JPEG et commentez l'effet de cette quantification sur un bloc 8 x 8.
2. Quelles conséquences attendre de cette quantification appliquée à une image préalablement transformée par une DCT ?
3. Ecrire les analogues des fonctions `quantifier()` et `dequantifier()` scalaires pour la quantification par une matrice de quantification quelconque. On l'appliquera d'abord avec la matrice $Q$ de la quantification JPEG.
4. Vérifier vos traitements avec [l'exemple de wikipedia](https://fr.wikipedia.org/wiki/JPEG), section "Transformée DCT" équation 5 $\to$ équation 9.

In [16]:
# VOTRE CODE ICI

## Exercice 4 : codages zig-zag et RLE

1. Ecrire les fonctions qui réalisent le [codage zig-zag](https://fr.wikipedia.org/wiki/JPEG#Codage,_compression_RLE_et_Huffman), puis le _codage pseudo-RLE_ des suites de zéros finaux obtenues.
2. Appliquer ces fonctions pour transformer un bloc 8x8 en un _court_ vecteur.
3. Appliquer ces codages sur le résultat d'exercice 3 et observer la compression ainsi obtenue.

In [16]:
# VOTRE CODE ICI

## Étape projet

**Livrable séance 6 :** 

_Première étape :_

- pipeline de compression "DCT par blocs puis quantification uniforme avec pas variable"
- variation du pas de quantification, mesures de MSE/PSNR et galerie de résultats.

_Seconde étape pré-projet :_

- pipeline de compression "mini-JPEG"
- pipeline de reconstruction "mini-JPEG"
- mesures de MSE/PSNR/facteurs de compression et galerie de résultats.